# 05 X打分
旧版规则匹配复现。X是促销规则菜单中已记录类别的丰富度，共11类；不代表评论者已实际接触或使用规则。

建议Python 3.12，首次在本目录安装requirements.txt。所有路径相对本项目。new采集目录不会被读取。

首次使用：在python目录的终端执行 python -m pip install -r requirements.txt，然后执行 jupyter lab。

数据范围：旧样本共22,831条，来自B站11,475条、抖音10,910条、小红书446条；本次旧样本没有微博。01_原始数据中的H文件是历史整理源文本节选，A文件是原采集记录的脱敏节选。小红书16条已核对评论数组一致，但对应检查点的两个采集元数据字段曾规范化；抖音3条分析正文按旧处理去除了末尾“作者赞过”UI徽标。

规则来源共111个，保留47个来源的49份已存正文；其余64个来源没有完整正文，保留可恢复的链接、描述或短引文及人工核对字段。材料可得性在来源表中单列，不把摘要当全文。

In [1]:
from pathlib import Path
import sys
# 从python目录或项目根启动均可；没有固定机器路径
PYTHON_DIR = Path.cwd() if (Path.cwd() / 'analysis_helpers.py').exists() else Path.cwd() / 'python'
if not (PYTHON_DIR / 'analysis_helpers.py').exists():
    raise FileNotFoundError('请从lasso论文或lasso论文/python目录打开此Notebook')
sys.path.insert(0, str(PYTHON_DIR.resolve()))
try:
    import analysis_helpers as h
except ImportError as exc:
    raise ImportError("缺少依赖，请在python目录运行：python -m pip install -r requirements.txt；安装后重启内核") from exc
print(h.runtime_check())

{'python': '3.12.14', 'numpy': '2.3.5', 'scipy': '1.17.0', 'scikit-learn': '1.8.0'}


## 1 读取已核对的规则材料
规则文本→语义事实→类别、来源等级与D1–D4材料覆盖，是历史人工核对步骤。程序从保留的实际规则CSV计算类别状态、材料门槛和X，不把旧X数值当作计算输入。未知不等于现实不存在；旧缺失匹配保持缺失。

In [2]:
x_rows, environment = h.compute_x()
print('全池:', len(x_rows), '有X:', int(x_rows.X.notna().sum()), '实际ENV:', x_rows.loc[x_rows.X.notna(), 'ENV'].nunique())
display(environment.head(8))

全池: 22831 有X: 2623 实际ENV: 27


,ENV,trading_platform,event,event_year,X_ENV,observed_category_count,unknown_category_count,document_gate_pass,source_admission_pass,coverage_rate,...,C2,C3,C4,C5,R1,R2,R3,R4,R5,R6
0,京东|双11|2022,京东,双11,2022,1.000000,11,0,True,True,1.000,...,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes
1,京东|双11|2023,京东,双11,2023,0.909091,10,1,True,True,1.000,...,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,unknown,observed_yes,observed_yes
2,淘宝|618|2021,淘宝,618,2021,0.727273,8,3,True,True,0.875,...,observed_yes,observed_yes,unknown,unknown,observed_yes,observed_yes,observed_yes,unknown,observed_yes,observed_yes
3,淘宝|双11|2021,淘宝,双11,2021,0.727273,8,3,True,True,0.750,...,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,unknown,unknown,unknown,observed_yes
4,京东|618|2023,京东,618,2023,1.000000,11,0,True,True,1.000,...,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes
5,淘宝|618|2023,淘宝,618,2023,0.636364,7,4,True,True,0.750,...,observed_yes,observed_yes,unknown,unknown,observed_yes,observed_yes,unknown,observed_yes,observed_yes,unknown
6,淘宝|618|2025,淘宝,618,2025,0.909091,10,1,True,True,0.750,...,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,observed_yes,unknown
7,淘宝|双11|2025,淘宝,双11,2025,0.727273,8,3,True,True,0.750,...,observed_yes,observed_yes,observed_yes,observed_yes,unknown,observed_yes,observed_yes,observed_yes,unknown,unknown


## 2 查看匹配结果
先按历史已核对的交易平台×活动×年份匹配ENV，再使用规则计算结果。信息来源平台（B站等）不等于交易平台（京东等）。只恢复旧准入记录，不自动为旧缺失记录补值。

In [3]:
display(x_rows.loc[x_rows.X.notna(), ['row_id','ENV','X']].head(10))
print(x_rows.groupby('stream').X.count().to_dict())

,row_id,ENV,X
0,BILI:BV11ZGX64EDa:checkpoint089_capture_1,抖音|618|2026,0.636364
1,BILI:BV11ZGX64EDa:checkpoint089_capture_10,抖音|618|2026,0.636364
2,BILI:BV11ZGX64EDa:checkpoint089_capture_11,抖音|618|2026,0.636364
3,BILI:BV11ZGX64EDa:checkpoint089_capture_12,抖音|618|2026,0.636364
5,BILI:BV11ZGX64EDa:checkpoint089_capture_14,抖音|618|2026,0.636364
6,BILI:BV11ZGX64EDa:checkpoint089_capture_15,抖音|618|2026,0.636364
7,BILI:BV11ZGX64EDa:checkpoint089_capture_16,抖音|618|2026,0.636364
8,BILI:BV11ZGX64EDa:checkpoint089_capture_17,抖音|618|2026,0.636364
9,BILI:BV11ZGX64EDa:checkpoint089_capture_18,抖音|618|2026,0.636364
10,BILI:BV11ZGX64EDa:checkpoint089_capture_19,抖音|618|2026,0.636364


{'A': 1919, 'H': 704}


## 3 可选导出
默认只校验和显示。开启后结果写回02_X匹配数据，不覆盖旧匹配元数据。

In [4]:
WRITE_OUTPUT = False
if WRITE_OUTPUT:
    output = h.DATA / '02_X匹配数据' / 'X重新计算.csv'
    x_rows.to_csv(output, index=False, encoding='utf-8-sig')
    print(output.relative_to(h.ROOT))